# RAG Pipeline Evaluation

RAGAS metrics + retrieval benchmarks. Run after ingesting documents.

**Metrics:** faithfulness, answer_relevancy, context_precision, context_recall, MRR@10

In [ ]:
import sys, os
sys.path.insert(0, '..')
from dotenv import load_dotenv
load_dotenv('../.env')
os.environ['ENVIRONMENT'] = 'dev'

import asyncio, json
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import nest_asyncio
nest_asyncio.apply()
print('Imports OK')

In [ ]:
# ── Build RAG chain ─────────────────────────────────────────────────────
from pathlib import Path
from src.retrieval.vector_store.faiss_vs import FAISSVectorStore
from src.retrieval.bm25_retriever import BM25Retriever
from src.retrieval.hybrid_retriever import HybridRetriever
from src.ingestion.embedders.openai_embedder import OpenAIEmbedder
from src.generation.rag_chain import RAGChain

FAISS_PATH = '../data/faiss_index'
BM25_PATH  = '../data/bm25_index'

vs = FAISSVectorStore(embedding_dim=3072)
if Path(FAISS_PATH).exists():
    vs.load(FAISS_PATH)
    print(f'FAISS loaded: {vs.total_vectors} vectors')
else:
    print('WARNING: No FAISS index found — run ingestion first')

bm25 = BM25Retriever.load(BM25_PATH) if Path(f'{BM25_PATH}/bm25.pkl').exists() \
       else BM25Retriever(corpus=[])
print(f'BM25: {len(bm25.corpus)} documents')

embedder  = OpenAIEmbedder()
retriever = HybridRetriever(vector_store=vs, bm25_retriever=bm25)
chain     = RAGChain.from_config(retriever=retriever, embedder=embedder)
print('RAG chain ready')

In [ ]:
# ── RAGAS evaluation ────────────────────────────────────────────────────
from src.evaluation.ragas_evaluator import RAGASEvaluator

evaluator = RAGASEvaluator(
    experiment_name='rag_evaluation',
    mlflow_tracking_uri='sqlite:///mlflow.db',
)
scores_df = evaluator.evaluate_from_file(
    qa_path='../data/eval/qa_pairs.json',
    rag_chain=chain,
    run_name='hybrid_rrf_gpt4o',
    batch_size=4,
)
scores_df.describe()

In [ ]:
# ── Retrieval benchmarks (dense vs sparse vs hybrid) ─────────────────────
from src.evaluation.metrics import evaluate_retrieval
import numpy as np

with open('../data/eval/qa_pairs.json') as f:
    qa = json.load(f)

questions = [d['question'] for d in qa]
gt_texts  = [d['ground_truth'] for d in qa]

# Embed all queries
q_embs = embedder.embed_documents(questions)   # reuse embed_documents for batch

# Approximate relevant chunk IDs using BM25 against ground truth
relevant_sets = [
    {r['chunk_id'] for r in bm25.retrieve(gt, k=3)}
    for gt in gt_texts
]

K = 10
strategies = {}

# Dense only
dense_retrieved = [
    [r['chunk_id'] for r in vs.similarity_search(emb, k=K) if isinstance(r, dict)]
    for emb in q_embs
]
strategies['Dense'] = evaluate_retrieval(questions, dense_retrieved, relevant_sets, k=K)

# Sparse only
sparse_retrieved = [
    [r['chunk_id'] for r in bm25.retrieve(q, k=K)]
    for q in questions
]
strategies['Sparse (BM25)'] = evaluate_retrieval(questions, sparse_retrieved, relevant_sets, k=K)

# Hybrid RRF
hybrid_retrieved = [
    [r.chunk_id for r in retriever.retrieve(emb, q, top_k_final=K)]
    for emb, q in zip(q_embs, questions)
]
strategies['Hybrid (RRF)'] = evaluate_retrieval(questions, hybrid_retrieved, relevant_sets, k=K)

for name, res in strategies.items():
    print(f'\n{name}:')
    res.print_report()

In [ ]:
# ── Visualize RAGAS distributions ───────────────────────────────────────
metric_cols = [c for c in ['faithfulness','answer_relevancy','context_precision','context_recall']
               if c in scores_df.columns]

fig, axes = plt.subplots(1, len(metric_cols), figsize=(5*len(metric_cols), 4))
if len(metric_cols)==1: axes=[axes]

for ax, col in zip(axes, metric_cols):
    scores_df[col].hist(bins=8, ax=ax, color='#4A90D9', edgecolor='white')
    mean_val = scores_df[col].mean()
    ax.axvline(mean_val, color='#E8593C', lw=2, ls='--', label=f'μ={mean_val:.3f}')
    ax.set_title(col.replace('_',' ').title(), fontsize=11)
    ax.set_xlim(0,1)
    ax.legend(fontsize=9)

plt.suptitle('RAGAS Score Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/eval/ragas_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to data/eval/ragas_distributions.png')

In [ ]:
# ── Summary table ────────────────────────────────────────────────────────
retrieval_df = pd.DataFrame({n: r.to_dict() for n,r in strategies.items()}).T
print('\nRetrieval comparison:')
print(retrieval_df.to_string())

if metric_cols:
    ragas_summary = scores_df[metric_cols].agg(['mean','std','min','max']).T.round(4)
    print('\nRAGAS summary:')
    print(ragas_summary.to_string())